# Stage 3 – Modelling

Leakage-aware preprocessing on a 200k sample, then Logistic Regression, Random Forest, XGBoost, HistGradientBoosting and a stacking ensemble.

**v2 fix:** the train/test split now happens *before* target encoding (out-of-fold encoding on train, train-only means for test), and Optuna tunes on a validation set carved from train. The first version encoded categories using the whole dataset, which leaked the target into the test set and inflated ROC AUC to 0.91.


In [2]:
# 03_modeling.ipynb
# ===========================================================
# Stage 3: Credit Risk Modeling + XAI (Leakage-aware, Memory-safe)
# v2: split before target encoding; hyperparameters tuned on a validation set
# Project: Explainable AI for Credit Risk Prediction (Sai)
# Data: accepted_cleaned.csv from Notebook 1
# ===========================================================
# LIME for local explanations

from lime.lime_tabular import LimeTabularExplainer
import os
import gc
import random
import warnings

from typing import List

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

# Modeling / preprocessing
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OrdinalEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier, StackingClassifier
from sklearn.metrics import (
    classification_report,
    roc_auc_score,
    average_precision_score,
    confusion_matrix,
    precision_recall_curve,
    auc,
)

import optuna
import matplotlib.pyplot as plt
import joblib
import xgboost as xgb
import shap

warnings.filterwarnings("ignore")
tqdm.pandas()

# -----------------------------------------------------------
# CONFIG
# -----------------------------------------------------------
# Cleaned dataset from Notebook 1. The first path that exists is used.
DATA_CANDIDATES = [
    "../data/accepted_cleaned.csv",                                   # repo layout
    r"D:\credit-risk-xai\credit-risk-xai\data\accepted_cleaned.csv",  # my local copy
]
INPUT_FILE = next((p for p in DATA_CANDIDATES if os.path.exists(p)), DATA_CANDIDATES[0])
print("Using data file:", INPUT_FILE)
SAMPLE_TARGET = 200_000                      # rows to sample (None => full file, but heavy)
RANDOM_STATE = 42
TEST_SIZE = 0.20
N_FOLDS_FOR_TE = 5                           # for target-mean encoding
OPTUNA_TRIALS_RF = 8
OPTUNA_TRIALS_XGB = 8
SAVE_DIR = "../outputs/notebook3_outputs"
os.makedirs(SAVE_DIR, exist_ok=True)

TARGET_COL = "default"   # created in Notebook 1: 1 = default, 0 = fully paid

# -----------------------------------------------------------
# Utility: memory-safe reservoir sampling from CSV
# -----------------------------------------------------------
def reservoir_sample_csv(
    path: str,
    sample_size: int,
    random_state: int = 42,
    chunksize: int = 100_000,
):
    """
    Reservoir sampling for large CSV.
    Returns: sampled_df (pandas), total_rows_read (int)
    """
    rng = random.Random(random_state)
    reservoir = []
    total = 0

    reader = pd.read_csv(
        path,
        compression="infer",
        chunksize=chunksize,
        low_memory=True
    )

    for chunk in tqdm(reader, desc="Streaming CSV for sampling"):
        for _, row in chunk.iterrows():
            if total < sample_size:
                reservoir.append(row)
            else:
                j = rng.randrange(total + 1)
                if j < sample_size:
                    reservoir[j] = row
            total += 1
        del chunk
        gc.collect()

    if total == 0:
        raise FileNotFoundError(f"No rows read from {path}")

    sampled_df = pd.DataFrame(reservoir).reset_index(drop=True)
    return sampled_df, total

# -----------------------------------------------------------
# 1) Sampling / loading dataset
# -----------------------------------------------------------
print("1) Sampling/loading dataset...")

if not os.path.exists(INPUT_FILE):
    raise FileNotFoundError(f"Input file not found: {INPUT_FILE}")

if SAMPLE_TARGET is not None:
    df, total_rows = reservoir_sample_csv(
        INPUT_FILE,
        sample_size=SAMPLE_TARGET,
        random_state=RANDOM_STATE,
    )
    print(f"Sampled {len(df)} rows from {total_rows} total rows.")
else:
    df = pd.read_csv(INPUT_FILE, low_memory=True)
    total_rows = len(df)
    print(f"Loaded full cleaned dataset: {df.shape}")

# -----------------------------------------------------------
# 2) Create binary target & drop leakage columns
# -----------------------------------------------------------
print("2) Create binary target and drop leakage cols if present...")

# Ensure target exists
if TARGET_COL not in df.columns:
    raise ValueError(f"Target column '{TARGET_COL}' not found in cleaned dataset.")

# Coerce target to int (0/1)
df[TARGET_COL] = pd.to_numeric(df[TARGET_COL], errors="coerce").fillna(0).astype(int)

# Explicit leakage columns: payment/outcome info etc.
leak_cols_explicit = [
    "loan_status",              # original text outcome (categorical)
    "total_pymnt",
    "total_pymnt_inv",
    "total_rec_prncp",
    "total_rec_int",
    "total_rec_late_fee",
    "last_pymnt_amnt",
    "recoveries",
    "collection_recovery_fee",
    "last_pymnt_d",
    "next_pymnt_d",
    "last_credit_pull_d",
    "out_prncp",
    "out_prncp_inv",
    "settlement_status",
    "settlement_date",
    "settlement_amount",
    "settlement_percentage",
    # ❗ Not available at application time / outcome-related
    "debt_settlement_flag",
]

# FICO at end of loan – future info
leak_cols_explicit += [
    "last_fico_range_low",
    "last_fico_range_high",
]

# Pure identifiers / text / region that we don't want as features
leak_cols_explicit += [
    "id",          # ❗ drop ID as feature
    "desc",
    "url",
    "title",
    "zip_code",
    "addr_state",
    "policy_code",
]


drop_now = [c for c in leak_cols_explicit if c in df.columns]
if drop_now:
    df.drop(columns=drop_now, inplace=True)
    print(f"--> Dropped explicit leakage columns: {drop_now}")

# Drop constant/all-null columns
nunique = df.nunique(dropna=True)
const_cols = [c for c in df.columns if nunique.get(c, 0) <= 1 and c != TARGET_COL]
if const_cols:
    df.drop(columns=const_cols, inplace=True)
    print(f"--> Dropped constant/all-null columns: {const_cols}")

print("Remaining columns:", len(df.columns))

# -----------------------------------------------------------
# 3) Detect numeric vs categorical features (robust)
# -----------------------------------------------------------
print("3) Detect numeric vs categorical features (robust detection)...")

# List candidate feature columns (exclude target)
feature_cols = [c for c in df.columns if c != TARGET_COL]

numeric_cols: List[str] = []
categorical_cols: List[str] = []

for c in feature_cols:
    # Take a sample to inspect type
    sample = df[c].dropna().head(500)
    # Try numeric conversion
    sample_num = pd.to_numeric(sample, errors="coerce")
    non_na = len(sample_num.dropna())
    frac_numeric = non_na / max(len(sample), 1)

    # Heuristic:
    # - mostly numeric values => numeric
    # - or high cardinality numeric-like strings
    if frac_numeric > 0.8:
        numeric_cols.append(c)
    else:
        categorical_cols.append(c)

print(f"Detected numeric columns: {len(numeric_cols)}, categorical columns: {len(categorical_cols)}")
print("Numeric examples:", numeric_cols[:10])
print("Categorical examples:", categorical_cols[:10])

# -----------------------------------------------------------
# 4) Coerce numeric columns & clean categorical
# -----------------------------------------------------------
print("4) Coercing numeric columns & cleaning categoricals...")

for c in numeric_cols:
    df[c] = pd.to_numeric(df[c], errors="coerce")

# For categorical: fill NaN with token and cast to string
if categorical_cols:
    df[categorical_cols] = df[categorical_cols].astype(str).fillna("__MISSING__")

# -----------------------------------------------------------
# 5) Split categorical by cardinality
# -----------------------------------------------------------
print("5) Splitting categorical columns by cardinality...")

small_cat = []
large_cat = []
for c in categorical_cols:
    nun = df[c].nunique()
    if nun < 20:
        small_cat.append(c)
    else:
        large_cat.append(c)

print(f"Small-cardinality: {len(small_cat)}  Large-cardinality: {len(large_cat)}")

# If too many large-cat, restrict to top-freq columns
MAX_LARGE_USE = 8
if len(large_cat) > MAX_LARGE_USE:
    # keep those with highest frequency mass in top-100 categories
    freq_sorted = sorted(
        large_cat,
        key=lambda col: df[col].value_counts().iloc[:100].sum(),
        reverse=True
    )
    keep_large = freq_sorted[:MAX_LARGE_USE]
    drop_large_from_te = [c for c in large_cat if c not in keep_large]
    print(f"Reducing large-cardinality columns for TE to top {MAX_LARGE_USE}. Ignoring TE on: {drop_large_from_te}")
    large_cat = keep_large
    # The dropped large-card cols remain in 'small_cat' with truncated labels
    for c in drop_large_from_te:
        df[c] = df[c].apply(lambda s: s if len(s) <= 30 else s[:30])
        small_cat.append(c)

###############################################################################
# 6. Train/test split FIRST (before any target-based encoding)
#    Fix: previously the split happened after target encoding, so category
#    means were computed using test rows too (target leakage).
###############################################################################
print("6) Train/test split (stratified) BEFORE encoding...")

train_df, test_df = train_test_split(
    df,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=df[TARGET_COL],
)
train_df = train_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)
y_train = train_df[TARGET_COL].astype(int)
y_test = test_df[TARGET_COL].astype(int)
print(f"Train rows: {len(train_df)}  Test rows: {len(test_df)}")

###############################################################################
# 7. Target-mean encoding, fitted on TRAIN only
#    - train rows: out-of-fold means (a row never sees its own label)
#    - test rows: means learned from the full training set
#    - smoothing pulls rare categories towards the global mean
###############################################################################
print("7) Leak-free target encoding (out-of-fold on train, train means on test)...")

TE_SMOOTHING = 20
global_mean = y_train.mean()

def smoothed_means(frame, col):
    stats = frame.groupby(col)[TARGET_COL].agg(["mean", "count"])
    return (stats["mean"] * stats["count"] + global_mean * TE_SMOOTHING) / (stats["count"] + TE_SMOOTHING)

X_te_train = pd.DataFrame(index=train_df.index)
X_te_test = pd.DataFrame(index=test_df.index)
te_encoders = {}

skf = StratifiedKFold(n_splits=N_FOLDS_FOR_TE, shuffle=True, random_state=RANDOM_STATE)
for col in large_cat:
    print(f"  TE -> {col}")
    oof = pd.Series(global_mean, index=train_df.index, dtype=float)
    for fit_idx, enc_idx in skf.split(train_df, y_train):
        means = smoothed_means(train_df.iloc[fit_idx], col)
        oof.iloc[enc_idx] = train_df[col].iloc[enc_idx].map(means).fillna(global_mean).values
    X_te_train[col + "_te"] = oof.values

    full_means = smoothed_means(train_df, col)
    te_encoders[col] = full_means.to_dict()
    X_te_test[col + "_te"] = test_df[col].map(full_means).fillna(global_mean).values

###############################################################################
# 8. Ordinal-encode small-cardinality categoricals (fit on TRAIN only)
###############################################################################
print("8) Ordinal-encoding small-cardinality categorical features (fit on train)...")

if small_cat:
    ordinal_encoder = OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)
    ord_names = [c + "_ord" for c in small_cat]
    X_ord_train = pd.DataFrame(ordinal_encoder.fit_transform(train_df[small_cat].astype(str)),
                               columns=ord_names, index=train_df.index)
    X_ord_test = pd.DataFrame(ordinal_encoder.transform(test_df[small_cat].astype(str)),
                              columns=ord_names, index=test_df.index)
else:
    ordinal_encoder = None
    X_ord_train = pd.DataFrame(index=train_df.index)
    X_ord_test = pd.DataFrame(index=test_df.index)

###############################################################################
# 9. Assemble train / test feature matrices
###############################################################################
print("9) Assembling feature matrices...")

numeric_cols_final = [c for c in numeric_cols if c in df.columns and c != TARGET_COL]

X_train = pd.concat([train_df[numeric_cols_final], X_ord_train, X_te_train], axis=1)
X_test = pd.concat([test_df[numeric_cols_final], X_ord_test, X_te_test], axis=1)
X_all = X_train  # kept so later cleanup code still works
feature_names = X_train.columns.tolist()
small_cat_used = list(small_cat)
large_cat_used = list(large_cat)

print(f"Train: {X_train.shape} Test: {X_test.shape}")

# -----------------------------------------------------------
# 10) Impute numeric features and scale
# -----------------------------------------------------------
print("10) Impute numeric features and scale (fit on train only)...")

preprocessor_num = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

if len(numeric_cols_final) > 0:
    X_train_num = preprocessor_num.fit_transform(X_train[numeric_cols_final])
    X_test_num = preprocessor_num.transform(X_test[numeric_cols_final])
else:
    # no numeric features
    X_train_num = np.empty((X_train.shape[0], 0))
    X_test_num = np.empty((X_test.shape[0], 0))

# The remaining columns (ordinal + TE) are already numeric, with no NaN
other_cols = [c for c in X_all.columns if c not in numeric_cols_final]
X_train_other = X_train[other_cols].to_numpy(dtype=float)
X_test_other = X_test[other_cols].to_numpy(dtype=float)

# Final matrices
X_train_final = np.hstack([X_train_num, X_train_other])
X_test_final = np.hstack([X_test_num, X_test_other])
final_feature_names = list(numeric_cols_final) + other_cols

print("Final matrices shapes:", X_train_final.shape, X_test_final.shape)

# Cleanup big objects
del X_all, X_train_num, X_test_num, X_train_other, X_test_other
gc.collect()

# -----------------------------------------------------------
# 11) Evaluation helper
# -----------------------------------------------------------
def evaluate_model(model, X_te, y_te, name="Model"):
    y_pred = model.predict(X_te)
    report = classification_report(y_te, y_pred, digits=4)

    # Get probabilities
    if hasattr(model, "predict_proba"):
        probs = model.predict_proba(X_te)[:, 1]
    else:
        # fallback for models with decision_function
        scores = model.decision_function(X_te)
        probs = (scores - scores.min()) / (scores.max() - scores.min() + 1e-9)

    roc = roc_auc_score(y_te, probs)
    prec, rec, _ = precision_recall_curve(y_te, probs)
    pr_auc = auc(rec, prec)
    cm = confusion_matrix(y_te, y_pred)

    print(f"\n=== {name} ===")
    print(report)
    print("Confusion Matrix:\n", cm)
    print(f"ROC AUC: {roc:.6f}, PR AUC: {pr_auc:.6f}")

    return {
        "report": report,
        "roc_auc": roc,
        "pr_auc": pr_auc,
        "cm": cm,
    }

# -----------------------------------------------------------
# 12) Logistic Regression baseline
# -----------------------------------------------------------
print("11) Training Logistic Regression (baseline)...")

lr = LogisticRegression(
    max_iter=1000,
    class_weight="balanced",
    solver="saga",
    n_jobs=-1,
    random_state=RANDOM_STATE,
)
lr.fit(X_train_final, y_train)
lr_results = evaluate_model(lr, X_test_final, y_test, "Logistic Regression")

joblib.dump(lr, os.path.join(SAVE_DIR, "logreg.joblib"))

# -----------------------------------------------------------
# Validation split for tuning (carved from TRAIN, test stays untouched)
# -----------------------------------------------------------
X_tr_tune, X_val_tune, y_tr_tune, y_val_tune = train_test_split(
    X_train_final, y_train, test_size=0.2, random_state=RANDOM_STATE, stratify=y_train
)

# -----------------------------------------------------------
# 13) Random Forest + Optuna tuning
# -----------------------------------------------------------
print("12) Tuning RandomForest (Optuna light)...")

def rf_objective(trial):
    params = {
        "n_estimators": trial.suggest_int("n_estimators", 100, 300),
        "max_depth": trial.suggest_int("max_depth", 4, 14),
        "min_samples_split": trial.suggest_int("min_samples_split", 2, 20),
        "min_samples_leaf": trial.suggest_int("min_samples_leaf", 1, 10),
        "class_weight": "balanced",
        "n_jobs": -1,
        "random_state": RANDOM_STATE,
    }
    clf = RandomForestClassifier(**params)
    clf.fit(X_tr_tune, y_tr_tune)
    preds = clf.predict_proba(X_val_tune)[:, 1]
    return roc_auc_score(y_val_tune, preds)   # tuned on validation, not test

study_rf = optuna.create_study(direction="maximize")
study_rf.optimize(rf_objective, n_trials=OPTUNA_TRIALS_RF, show_progress_bar=True)
best_rf_params = study_rf.best_trial.params
print("Best RF params:", best_rf_params)

rf = RandomForestClassifier(
    **best_rf_params,
    class_weight="balanced",
    n_jobs=-1,
    random_state=RANDOM_STATE,
)
rf.fit(X_train_final, y_train)
rf_results = evaluate_model(rf, X_test_final, y_test, "Random Forest")

joblib.dump(rf, os.path.join(SAVE_DIR, "random_forest.joblib"))

# -----------------------------------------------------------
# 14) XGBoost + Optuna tuning
# -----------------------------------------------------------
print("13) Tuning XGBoost (Optuna light)...")

def xgb_objective(trial):
    params = {
        "objective": "binary:logistic",
        "eval_metric": "auc",
        "tree_method": "hist",
        "n_jobs": -1,
        "random_state": RANDOM_STATE,
        "eta": trial.suggest_float("eta", 0.01, 0.3),
        "max_depth": trial.suggest_int("max_depth", 3, 10),
        "subsample": trial.suggest_float("subsample", 0.5, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.5, 1.0),
        "n_estimators": trial.suggest_int("n_estimators", 100, 400),
    }
    clf = xgb.XGBClassifier(**params)
    clf.fit(X_tr_tune, y_tr_tune)
    preds = clf.predict_proba(X_val_tune)[:, 1]
    return roc_auc_score(y_val_tune, preds)   # tuned on validation, not test

study_xgb = optuna.create_study(direction="maximize")
study_xgb.optimize(xgb_objective, n_trials=OPTUNA_TRIALS_XGB, show_progress_bar=True)
best_xgb_params = study_xgb.best_trial.params
print("Best XGB params:", best_xgb_params)

xgb_clf = xgb.XGBClassifier(
    objective="binary:logistic",
    eval_metric="auc",
    tree_method="hist",
    n_jobs=-1,
    random_state=RANDOM_STATE,
    **best_xgb_params,
)
xgb_clf.fit(X_train_final, y_train)
xgb_results = evaluate_model(xgb_clf, X_test_final, y_test, "XGBoost")

joblib.dump(xgb_clf, os.path.join(SAVE_DIR, "xgboost.joblib"))

# -----------------------------------------------------------
# 15) HistGradientBoosting (fast, can handle NaNs)
# -----------------------------------------------------------
print("14) Training HistGradientBoostingClassifier (fast; accepts NaNs)...")

hgb = HistGradientBoostingClassifier(
    max_iter=200,
    random_state=RANDOM_STATE,
)
hgb.fit(X_train_final, y_train)
hgb_results = evaluate_model(hgb, X_test_final, y_test, "HistGradientBoosting")

joblib.dump(hgb, os.path.join(SAVE_DIR, "histgb.joblib"))

# -----------------------------------------------------------
# 16) Stacking ensemble
# -----------------------------------------------------------
print("15) Training stacking ensemble (light)...")

estimators = [
    ("rf", rf),
    ("xgb", xgb_clf),
    ("hgb", hgb),
]
stack = StackingClassifier(
    estimators=estimators,
    final_estimator=LogisticRegression(max_iter=1000, class_weight="balanced"),
    n_jobs=-1,
)
stack.fit(X_train_final, y_train)
stack_results = evaluate_model(stack, X_test_final, y_test, "Stacking Ensemble")

joblib.dump(stack, os.path.join(SAVE_DIR, "stacking.joblib"))

# -----------------------------------------------------------
# 17) SHAP explanations (XAI)
# -----------------------------------------------------------
print("16) SHAP explanations (sampled subset)...")

explainer_model = xgb_clf  # choose tree-based model for SHAP
shap_sample_size = min(2000, X_test_final.shape[0])
rng = np.random.RandomState(RANDOM_STATE)
shap_idx = rng.choice(np.arange(X_test_final.shape[0]), size=shap_sample_size, replace=False)

X_shap = X_test_final[shap_idx]
y_shap = y_test.iloc[shap_idx].values

explainer = shap.TreeExplainer(explainer_model)
shap_values = explainer.shap_values(X_shap)  # shape (n_samples, n_features)

mean_abs_shap = np.abs(shap_values).mean(axis=0)
shap_importance = sorted(
    zip(final_feature_names, mean_abs_shap),
    key=lambda x: x[1],
    reverse=True
)[:30]

print("Top 30 features by mean |SHAP| (no target / payment leakage):")
for feat, imp in shap_importance:
    print(f"  {feat}: {imp:.6f}")

joblib.dump(
    {"explainer": explainer, "feature_names": final_feature_names},
    os.path.join(SAVE_DIR, "shap_explainer.joblib")
)

# Example local explanation
print("\nExample local explanation for one test instance:")
print("True label:", int(y_shap[0]))
inst_shap = shap_values[0]
top_k = np.argsort(np.abs(inst_shap))[::-1][:12]
for i in top_k:
    print(f"  {final_feature_names[i]} => SHAP {inst_shap[i]:.6f}")

# Global SHAP summary (top features)
shap.summary_plot(shap_values, X_shap, feature_names=feature_names, show=False)
plt.tight_layout()
plt.savefig("../outputs/notebook3_outputs/shap_summary.png", dpi=300)
plt.close()

# Optional: bar plot (easier to read)
shap.summary_plot(shap_values, X_shap, feature_names=feature_names, plot_type="bar", show=False)
plt.tight_layout()
plt.savefig("../outputs/notebook3_outputs/shap_summary_bar.png", dpi=300)
plt.close()

# Local explanation for one instance
i = 0  # first sample in X_shap
shap.plots.waterfall(shap.Explanation(values=shap_values[i],
                                      base_values=explainer.expected_value,
                                      data=X_shap[i],
                                      feature_names=feature_names),
                     show=False)
plt.tight_layout()
plt.savefig("../outputs/notebook3_outputs/shap_local_example.png", dpi=300)
plt.close()

print("Saved SHAP plots to notebook3_outputs/")
#Lime 

import numpy as np

print("\n=== LIME: setting up explainer for XGBoost ===")

# 1) Training data for LIME (use a subset if needed to speed up)
lime_train = X_train_final
# If you want to speed up drastically:
# lime_train = X_train_final[:20000]

# 2) Class names (0 = non-default, 1 = default)
class_names = ["non_default", "default"]

# 3) Build the LimeTabularExplainer
lime_explainer = LimeTabularExplainer(
    training_data=lime_train,
    feature_names=feature_names,
    class_names=class_names,
    mode="classification",
    discretize_continuous=True
)

print("LIME explainer ready.")





import numpy as np
import matplotlib.pyplot as plt

# pick an instance index
instance_idx = 0   # change this to inspect different customers

# 1) raw instanceA
x_instance = X_test_final[instance_idx]

# 2) explain (ask for top 2 labels to be safe)
lime_exp = lime_explainer.explain_instance(
    data_row=x_instance,
    predict_fn=xgb_clf.predict_proba,
    num_features=10,
    top_labels=2    # get explanations for both 0 and 1 if needed
)

# model prediction
probs = xgb_clf.predict_proba(X_test_final[instance_idx:instance_idx+1])[0]
pred_label = int(np.argmax(probs))   # 0 or 1

print(f"True label: {int(y_test.iloc[instance_idx])}")
print(f"Predicted probabilities (non_default, default): {probs}")
print(f"Predicted class: {pred_label}")

# Which label to show?
# - if you want "whatever LIME has": use the predicted class
# - if you specifically want default (1), but it's not there, fall back to predicted
label_to_show = 1 if 1 in lime_exp.local_exp else pred_label

print(f"\nLIME explanation for label: {label_to_show} (0=non-default, 1=default)")
print("Feature contributions:")
for feature, weight in lime_exp.as_list(label=label_to_show):
    print(f"  {feature}: {weight:.4f}")

# Plot and save
fig = lime_exp.as_pyplot_figure(label=label_to_show)
plt.tight_layout()
plt.savefig("../outputs/notebook3_outputs/lime_local_example.png", dpi=300)
plt.close()

print("\nSaved LIME plot to notebook3_outputs/lime_local_example.png")


# --------------------------------------------------
# SAVE FULL PREPROCESSING PIPELINE (NUM + CAT + TE)
# --------------------------------------------------
preprocessing_bundle = {
    "numeric_preprocessor": preprocessor_num,   # SimpleImputer + StandardScaler
    "ordinal_encoder": ordinal_encoder,         # may be None if no small_cat
    "target_encoders": te_encoders,             # dict of {col: {category: mean}}
    "feature_names": feature_names,             # final model feature order
    "numeric_cols_final": numeric_cols_final,
    "small_cat_used": small_cat_used,
    "large_cat_used": large_cat_used,
}

pipeline_path = os.path.join(SAVE_DIR, "preprocessing_pipeline.joblib")
joblib.dump(preprocessing_bundle, pipeline_path)
print(f"\n✅ Full preprocessing pipeline saved to {pipeline_path}")

# -----------------------------------------------------------
# 18) Save cleaned modeling sample
# -----------------------------------------------------------
print("17) Saving cleaned sampled dataset & metadata (small CSV)...")

cleaned_sample = pd.DataFrame(X_test_final, columns=final_feature_names).reset_index(drop=True)
cleaned_sample[TARGET_COL] = y_test.reset_index(drop=True).values

cleaned_path = os.path.join(SAVE_DIR, "cleaned_sample_for_modelling.csv.gz")
cleaned_sample.to_csv(cleaned_path, index=False, compression="gzip")
print("Saved:", cleaned_path)

meta = {
    "feature_names": final_feature_names,
    "numeric_cols": numeric_cols_final,
    "large_cat_used": large_cat,
    "small_cat_used": small_cat,
}
joblib.dump(meta, os.path.join(SAVE_DIR, "feature_metadata.joblib"))
print("Saved metadata to:", os.path.join(SAVE_DIR, "feature_metadata.joblib"))

print("Done. All models and metadata saved to", SAVE_DIR)

# 18) Threshold tuning for better recall on defaulters

from sklearn.metrics import precision_recall_curve, classification_report, confusion_matrix

best_model = xgb_clf   # you can change to rf_clf, hgb_clf, stack_clf

probs = best_model.predict_proba(X_test_final)[:, 1]

precisions, recalls, thresholds = precision_recall_curve(y_test, probs)

def find_threshold_for_recall(target_recall, recalls, thresholds):
    diffs = np.abs(recalls[:-1] - target_recall)
    idx = diffs.argmin()
    return thresholds[idx], idx

target_recall = 0.50
best_thr, idx = find_threshold_for_recall(target_recall, recalls, thresholds)

print(f"\nChosen target recall: {target_recall:.2f}")
print(f"Closest recall: {recalls[idx]:.4f}")
print(f"Threshold: {best_thr:.4f}")
print(f"Precision: {precisions[idx]:.4f}")

y_pred_custom = (probs >= best_thr).astype(int)

print("\n=== Evaluation with tuned threshold ===")
print(classification_report(y_test, y_pred_custom, digits=4))
print("Confusion matrix:\n", confusion_matrix(y_test, y_pred_custom))

# Compare with default 0.5
y_pred_default = (probs >= 0.5).astype(int)
print("\n=== Default threshold (0.5) ===")
print(classification_report(y_test, y_pred_default, digits=4))
print("Confusion matrix:\n", confusion_matrix(y_test, y_pred_default))

from sklearn.metrics import roc_curve, precision_recall_curve, auc
import matplotlib.pyplot as plt

probs = xgb_clf.predict_proba(X_test_final)[:, 1]

# ROC
fpr, tpr, _ = roc_curve(y_test, probs)
roc_auc = roc_auc_score(y_test, probs)

plt.figure()
plt.plot(fpr, tpr, label=f"XGBoost (AUC = {roc_auc:.3f})")
plt.plot([0, 1], [0, 1], linestyle="--")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve - XGBoost")
plt.legend()
plt.tight_layout()
plt.savefig("../outputs/notebook3_outputs/xgb_roc_curve.png", dpi=300)
plt.close()

# Precision–Recall
prec, rec, _ = precision_recall_curve(y_test, probs)
pr_auc = auc(rec, prec)

plt.figure()
plt.plot(rec, prec, label=f"XGBoost (PR AUC = {pr_auc:.3f})")
plt.xlabel("Recall")
plt.ylabel("Precision")
plt.title("Precision–Recall Curve - XGBoost")
plt.legend()
plt.tight_layout()
plt.savefig("../outputs/notebook3_outputs/xgb_pr_curve.png", dpi=300)
plt.close()

print("Saved ROC and PR curves to notebook3_outputs/")


import pandas as pd

results_table = pd.DataFrame([
    ["Logistic Regression", lr_results["roc_auc"], lr_results["pr_auc"]],
    ["Random Forest",        rf_results["roc_auc"], rf_results["pr_auc"]],
    ["XGBoost",              xgb_results["roc_auc"], xgb_results["pr_auc"]],
    ["HistGradientBoosting", hgb_results["roc_auc"], hgb_results["pr_auc"]],
    ["Stacking Ensemble",    stack_results["roc_auc"], stack_results["pr_auc"]],
], columns=["Model", "ROC_AUC", "PR_AUC"])

print(results_table)
results_table.to_csv("../outputs/notebook3_outputs/model_comparison.csv", index=False)
print("\nSaved model comparison to notebook3_outputs/model_comparison.csv")


Using data file: D:\credit-risk-xai\credit-risk-xai\data\accepted_cleaned.csv
1) Sampling/loading dataset...


Streaming CSV for sampling: 14it [01:29,  6.36s/it]


Sampled 200000 rows from 1345310 total rows.
2) Create binary target and drop leakage cols if present...
--> Dropped explicit leakage columns: ['out_prncp', 'out_prncp_inv', 'debt_settlement_flag', 'last_fico_range_low', 'last_fico_range_high', 'id']
--> Dropped constant/all-null columns: ['hardship_flag']
Remaining columns: 86
3) Detect numeric vs categorical features (robust detection)...
Detected numeric columns: 74, categorical columns: 11
Numeric examples: ['loan_amnt', 'funded_amnt', 'funded_amnt_inv', 'int_rate', 'installment', 'annual_inc', 'dti', 'delinq_2yrs', 'fico_range_low', 'fico_range_high']
Categorical examples: ['term', 'grade', 'sub_grade', 'emp_title', 'emp_length', 'home_ownership', 'verification_status', 'purpose', 'earliest_cr_line', 'initial_list_status']
4) Coercing numeric columns & cleaning categoricals...
5) Splitting categorical columns by cardinality...
Small-cardinality: 8  Large-cardinality: 3
6) Train/test split (stratified) BEFORE encoding...
Train rows

[I 2026-09-29 14:10:14,608] A new study created in memory with name: no-name-a13c44e4-d184-498c-87d5-e3ee70cc5ebc


12) Tuning RandomForest (Optuna light)...


...

[I 2026-09-29 14:13:15,174] Trial 6 finished with value: 0.6787666097964264 and parameters: {'eta': 0.227383031516191, 'max_depth': 10, 'subsample': 0.5321117280272489, 'colsample_bytree': 0.9051874665796864, 'n_estimators': 199}. Best is trial 1 with value: 0.7269893117695989.


Best trial: 1. Best value: 0.726989: 100%|██████████| 8/8 [00:54<00:00,  6.82s/it]


[I 2026-09-29 14:13:20,673] Trial 7 finished with value: 0.702242300987528 and parameters: {'eta': 0.1726541018954486, 'max_depth': 9, 'subsample': 0.8519334681400446, 'colsample_bytree': 0.6716247134707012, 'n_estimators': 141}. Best is trial 1 with value: 0.7269893117695989.
Best XGB params: {'eta': 0.09979928125049552, 'max_depth': 3, 'subsample': 0.9902692868761568, 'colsample_bytree': 0.7696894320622029, 'n_estimators': 283}

=== XGBoost ===
              precision    recall  f1-score   support

           0     0.8131    0.9834    0.8902     32008
           1     0.5876    0.0948    0.1633      7992

    accuracy                         0.8058     40000
   macro avg     0.7004    0.5391    0.5268     40000
weighted avg     0.7681    0.8058    0.7450     40000

Confusion Matrix:
 [[31476   532]
 [ 7234   758]]
ROC AUC: 0.727372, PR AUC: 0.398984
14) Training HistGradientBoostingClassifier (fast; accepts NaNs)...


  File "d:\anaconda\envs\credit-risk-xai\lib\site-packages\joblib\externals\loky\backend\context.py", line 257, in _count_physical_cores
    cpu_info = subprocess.run(
  File "d:\anaconda\envs\credit-risk-xai\lib\subprocess.py", line 505, in run
    with Popen(*popenargs, **kwargs) as process:
  File "d:\anaconda\envs\credit-risk-xai\lib\subprocess.py", line 951, in __init__
    self._execute_child(args, executable, preexec_fn, close_fds,
  File "d:\anaconda\envs\credit-risk-xai\lib\subprocess.py", line 1436, in _execute_child
    hp, ht, pid, tid = _winapi.CreateProcess(executable, args,



=== HistGradientBoosting ===
              precision    recall  f1-score   support

           0     0.8124    0.9820    0.8892     32008
           1     0.5603    0.0918    0.1578      7992

    accuracy                         0.8042     40000
   macro avg     0.6864    0.5369    0.5235     40000
weighted avg     0.7620    0.8042    0.7431     40000

Confusion Matrix:
 [[31432   576]
 [ 7258   734]]
ROC AUC: 0.726065, PR AUC: 0.395545
15) Training stacking ensemble (light)...

=== Stacking Ensemble ===
              precision    recall  f1-score   support

           0     0.8826    0.6867    0.7724     32008
           1     0.3357    0.6341    0.4390      7992

    accuracy                         0.6762     40000
   macro avg     0.6092    0.6604    0.6057     40000
weighted avg     0.7733    0.6762    0.7058     40000

Confusion Matrix:
 [[21980 10028]
 [ 2924  5068]]
ROC AUC: 0.724874, PR AUC: 0.396756
16) SHAP explanations (sampled subset)...
Top 30 features by mean |SHAP| (n

In [3]:
import numpy as np
import matplotlib.pyplot as plt

# --------- choose an instance ----------
instance_idx = 0   # change this number to inspect different customers

# 1) Get the instance from X_test_final (works for both ndarray & DataFrame)
if isinstance(X_test_final, np.ndarray):
    x_instance = X_test_final[instance_idx]              # shape (n_features,)
else:
    x_instance = X_test_final.iloc[instance_idx].values  # convert to 1D array

# 2) LIME explanation
lime_exp = lime_explainer.explain_instance(
    data_row=x_instance,
    predict_fn=xgb_clf.predict_proba,
    num_features=10,
    top_labels=2    # ask for explanations for both classes
)

# 3) Model prediction for this borrower
probs = xgb_clf.predict_proba(x_instance.reshape(1, -1))[0]
pred_label = int(np.argmax(probs))

# ✅ use iloc here so we don’t get KeyError on index
true_label = int(y_test.iloc[instance_idx])

print(f"True label: {true_label}")
print(f"Predicted probabilities (class 0, class 1): {probs}")
print(f"Predicted class: {pred_label}")

# 4) Decide which label to explain (prefer default class = 1 if available)
if 1 in lime_exp.local_exp:
    label_to_show = 1
else:
    label_to_show = pred_label

print(f"\nShowing LIME explanation for label: {label_to_show}")
print("\nLIME explanation (feature contributions):")
for feature, weight in lime_exp.as_list(label=label_to_show):
    print(f"  {feature}: {weight:.4f}")

# 5) Plot & save for thesis
fig = lime_exp.as_pyplot_figure(label=label_to_show)
plt.tight_layout()
plt.savefig("../outputs/notebook3_outputs/lime_local_example.png", dpi=300)
plt.close()

print("\nSaved LIME plot to notebook3_outputs/lime_local_example.png")


True label: 0
Predicted probabilities (class 0, class 1): [0.9083114 0.0916886]
Predicted class: 0

Showing LIME explanation for label: 1

LIME explanation (feature contributions):
  term_ord > 0.00: 0.0759
  num_tl_30dpd <= -0.05: 0.0585
  0.11 < sub_grade_te <= 0.19: -0.0452
  dti <= -0.55: -0.0359
  grade_ord <= 1.00: -0.0351
  delinq_amnt <= -0.02: -0.0343
  home_ownership_ord <= 1.00: -0.0268
  installment <= -0.72: -0.0210
  -0.74 < funded_amnt <= -0.28: -0.0200
  purpose_ord > 2.00: 0.0194

Saved LIME plot to notebook3_outputs/lime_local_example.png
